# DeiT Attention Maps — BreaKHis 80:10:10 (Google Colab)

This notebook loads the **already-trained `deit_best.pth`** checkpoint, recreates the same patient-level 80:10:10 split (seed 42), and generates DeiT class-token attention maps for test images. **It does not retrain the model.**

Before running: select **Runtime → Change runtime type → T4 GPU** (or another GPU). The dataset must be available under `/content/breakhis_dataset` (or update `DATA_DIR` below). Checkpoint path is set to your Google Drive folder. The maps and CSV metadata are saved to Drive.

The visualisation uses the last DeiT block's query-key attention, averaged over heads. This is an attention visualisation, not a clinical explanation or proof of which tissue features caused the prediction.


In [ ]:
# 1. Mount Drive and set paths
from google.colab import drive
from pathlib import Path
import os, re, random
import numpy as np
import pandas as pd
import torch
import timm
import cv2
import matplotlib.pyplot as plt
from PIL import Image
from torchvision import transforms
from torch.utils.data import Dataset
from sklearn.model_selection import StratifiedGroupKFold
from tqdm.auto import tqdm

drive.mount('/content/drive')

DATA_DIR = Path('/content/breakhis_dataset')  # Change this if your dataset is elsewhere
SAVE_DIR = Path('/content/drive/MyDrive/Breast_Cancer_Research/breakhis_savedmodels/breakhis_80_10_10')
WEIGHTS_PATH = SAVE_DIR / 'deit_best.pth'
OUTPUT_DIR = SAVE_DIR / 'deit_attention'

for folder_name in ['original', 'heatmap', 'overlay', 'Benign', 'Malignant']:
    (OUTPUT_DIR / folder_name).mkdir(parents=True, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
if device.type != 'cuda':
    print('WARNING: GPU not detected. In Colab, choose Runtime > Change runtime type > GPU.')
print('Checkpoint:', WEIGHTS_PATH)
print('Dataset directory:', DATA_DIR)
print('Output directory:', OUTPUT_DIR)
assert WEIGHTS_PATH.exists(), f'Checkpoint not found: {WEIGHTS_PATH}'
assert DATA_DIR.exists(), f'Dataset directory not found: {DATA_DIR}. Set DATA_DIR to the extracted BreaKHis folder.'


In [ ]:
# 2. Recreate the same patient-level 80:10:10 split used in the training notebook
IMAGE_EXTS = {'.png', '.jpg', '.jpeg', '.tif', '.tiff'}

def parse_breakhis_label(path):
    name = path.name.lower()
    parts = [part.lower() for part in path.parts]
    if any('malignant' in part for part in parts) or name.startswith('sob_m_') or '_m_' in name:
        return 1
    if any('benign' in part for part in parts) or name.startswith('sob_b_') or '_b_' in name:
        return 0
    return None

def extract_patient_id(path):
    stem = path.stem
    match = re.match(r'^(SOB_[A-Za-z]_[A-Za-z]+-\d+-[^-]+)-\d+-\d+$', stem)
    if match:
        return match.group(1)
    parts = stem.split('-')
    if len(parts) >= 3 and parts[-1].isdigit() and parts[-2].isdigit():
        return '-'.join(parts[:-2])
    return path.parent.name

rows = []
for p in DATA_DIR.rglob('*'):
    if p.is_file() and p.suffix.lower() in IMAGE_EXTS:
        label = parse_breakhis_label(p)
        if label is not None:
            rows.append({'path': str(p), 'label': int(label), 'patient_id': extract_patient_id(p), 'filename': p.name})

df = pd.DataFrame(rows).drop_duplicates(subset=['filename']).reset_index(drop=True)
if df.empty:
    raise ValueError('No BreaKHis images found. Check DATA_DIR and dataset folder structure.')

patient_df = df.drop_duplicates(subset=['patient_id']).reset_index(drop=True)
sgkf = StratifiedGroupKFold(n_splits=10, shuffle=True, random_state=42)
splits = list(sgkf.split(patient_df, patient_df['label'], groups=patient_df['patient_id']))
val_patients = set(patient_df.iloc[splits[0][1]]['patient_id'])
test_patients = set(patient_df.iloc[splits[1][1]]['patient_id'])
test_df = df[df['patient_id'].isin(test_patients)].reset_index(drop=True)

print(f'All images: {len(df):,} | patients: {patient_df.patient_id.nunique():,}')
print(f'Test images: {len(test_df):,} | benign: {(test_df.label == 0).sum():,} | malignant: {(test_df.label == 1).sum():,}')
print('Important: this must recreate the same split as the run that produced deit_best.pth.')


In [ ]:
# 3. Create evaluation dataset and load the trained DeiT checkpoint
IMG_SIZE = 224
mean = [0.485, 0.456, 0.406]
std = [0.229, 0.224, 0.225]
eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std),
])

class BreaKHisDataset(Dataset):
    def __init__(self, frame, transform=None):
        self.df = frame.reset_index(drop=True)
        self.transform = transform
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row['path']).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, int(row['label'])

test_dataset = BreaKHisDataset(test_df, eval_transform)

model = timm.create_model('deit_base_patch16_224', pretrained=False, num_classes=2)
checkpoint = torch.load(WEIGHTS_PATH, map_location='cpu')
if isinstance(checkpoint, dict) and 'state_dict' in checkpoint:
    state_dict = checkpoint['state_dict']
elif isinstance(checkpoint, dict) and 'model_state_dict' in checkpoint:
    state_dict = checkpoint['model_state_dict']
else:
    state_dict = checkpoint
state_dict = {k.removeprefix('module.'): v for k, v in state_dict.items()}
model.load_state_dict(state_dict, strict=True)
model = model.to(device).eval()
print('DeiT checkpoint loaded successfully.')


In [ ]:
# 4. Extract last-block DeiT attention from Q and K
# DeiT-Base has 12 attention heads. Read this from the model rather than hard-coding it.
attention_cache = {}
last_attn = model.blocks[-1].attn

def save_qkv(module, inputs, output):
    attention_cache['qkv'] = output.detach()

hook_handle = last_attn.qkv.register_forward_hook(save_qkv)

def get_deit_attention(input_tensor):
    """Return last-block CLS-to-patch attention, resized to a 14x14 patch grid."""
    attention_cache.clear()
    with torch.no_grad():
        logits = model(input_tensor)
    qkv = attention_cache.get('qkv')
    if qkv is None:
        raise RuntimeError('Could not capture DeiT QKV values. Check the timm model version.')
    batch, tokens, three_dim = qkv.shape
    heads = last_attn.num_heads
    head_dim = three_dim // 3 // heads
    qkv = qkv.reshape(batch, tokens, 3, heads, head_dim).permute(2, 0, 3, 1, 4)
    q, k = qkv[0], qkv[1]
    scale = getattr(last_attn, 'scale', head_dim ** -0.5)
    scores = (q @ k.transpose(-2, -1)) * scale
    attn = scores.softmax(dim=-1).mean(dim=1)  # average heads: [B, tokens, tokens]
    # DeiT token 0 is CLS; remaining 196 tokens correspond to 14x14 patches.
    cls_to_patches = attn[0, 0, 1:]
    grid_size = int(np.sqrt(cls_to_patches.numel()))
    if grid_size * grid_size != cls_to_patches.numel():
        raise ValueError(f'Unexpected DeiT patch token count: {cls_to_patches.numel()}')
    heat = cls_to_patches.reshape(grid_size, grid_size).cpu().numpy()
    heat = (heat - heat.min()) / (heat.max() - heat.min() + 1e-8)
    return logits, heat

print('Attention extraction ready. Last block:', len(model.blocks) - 1, '| heads:', last_attn.num_heads)


In [ ]:
# 5. Generate maps for 50 benign + 50 malignant test images
# Set either count lower if you want a quicker test run first.
N_BENIGN = 50
N_MALIGNANT = 50
RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)

benign_indices = np.flatnonzero(test_df['label'].to_numpy() == 0)
malignant_indices = np.flatnonzero(test_df['label'].to_numpy() == 1)
if len(benign_indices) < N_BENIGN or len(malignant_indices) < N_MALIGNANT:
    raise ValueError(f'Not enough test images. Benign available={len(benign_indices)}, malignant available={len(malignant_indices)}.')
selected_indices = list(rng.choice(benign_indices, N_BENIGN, replace=False)) + list(rng.choice(malignant_indices, N_MALIGNANT, replace=False))

metadata = []
for counter, dataset_idx in enumerate(tqdm(selected_indices, desc='Generating DeiT attention maps'), start=1):
    image_tensor, true_label = test_dataset[dataset_idx]
    row = test_df.iloc[dataset_idx]
    image_path = Path(row['path'])
    input_tensor = image_tensor.unsqueeze(0).to(device)
    logits, attention = get_deit_attention(input_tensor)
    probs = torch.softmax(logits, dim=1)[0]
    pred_label = int(probs.argmax().item())
    confidence = float(probs[pred_label].item())
    true_name = 'Benign' if int(true_label) == 0 else 'Malignant'
    pred_name = 'Benign' if pred_label == 0 else 'Malignant'

    original = Image.open(image_path).convert('RGB').resize((224, 224))
    original_np = np.asarray(original)
    attention_uint8 = np.uint8(255 * cv2.resize(attention, (224, 224), interpolation=cv2.INTER_CUBIC))
    heatmap = cv2.cvtColor(cv2.applyColorMap(attention_uint8, cv2.COLORMAP_JET), cv2.COLOR_BGR2RGB)
    overlay = cv2.addWeighted(original_np, 0.60, heatmap, 0.40, 0)

    base = f'{counter:03d}_{true_name}_{image_path.stem}_pred_{pred_name}_conf_{confidence*100:.2f}'
    paths = {
        'original_saved': OUTPUT_DIR / 'original' / f'{base}.png',
        'heatmap_saved': OUTPUT_DIR / 'heatmap' / f'{base}.png',
        'overlay_saved': OUTPUT_DIR / 'overlay' / f'{base}.png',
        'class_overlay_saved': OUTPUT_DIR / true_name / f'{base}.png',
    }
    Image.fromarray(original_np).save(paths['original_saved'])
    Image.fromarray(heatmap).save(paths['heatmap_saved'])
    Image.fromarray(overlay).save(paths['overlay_saved'])
    Image.fromarray(overlay).save(paths['class_overlay_saved'])

    metadata.append({
        'map_id': counter, 'dataset_index': int(dataset_idx), 'original_image': str(image_path),
        'true_class': true_name, 'predicted_class': pred_name,
        'confidence': confidence, 'confidence_percent': confidence * 100,
        'correct': int(true_label) == pred_label,
        'original_saved': str(paths['original_saved']), 'heatmap_saved': str(paths['heatmap_saved']),
        'overlay_saved': str(paths['overlay_saved']), 'class_overlay_saved': str(paths['class_overlay_saved'])
    })

hook_handle.remove()
metadata_df = pd.DataFrame(metadata)
metadata_path = OUTPUT_DIR / 'deit_100_attention_metadata.csv'
metadata_df.to_csv(metadata_path, index=False)
summary = metadata_df.groupby('true_class').agg(
    number_of_images=('map_id', 'count'),
    mean_confidence_percent=('confidence_percent', 'mean'),
    min_confidence_percent=('confidence_percent', 'min'),
    max_confidence_percent=('confidence_percent', 'max'),
    correct_predictions=('correct', 'sum')
).reset_index()
summary_path = OUTPUT_DIR / 'deit_100_attention_summary.csv'
summary.to_csv(summary_path, index=False)
print('Finished. Maps saved to:', OUTPUT_DIR)
print('Metadata:', metadata_path)
print('Summary:', summary_path)
display(summary)


In [ ]:
# 6. Preview a few generated examples
from IPython.display import display
preview = metadata_df.head(6)
fig, axes = plt.subplots(len(preview), 3, figsize=(12, 4 * len(preview)))
if len(preview) == 1:
    axes = np.expand_dims(axes, axis=0)
for i, (_, row) in enumerate(preview.iterrows()):
    axes[i, 0].imshow(Image.open(row['original_saved']))
    axes[i, 0].set_title(f"Original — {row['true_class']}")
    axes[i, 1].imshow(Image.open(row['heatmap_saved']))
    axes[i, 1].set_title('DeiT attention')
    axes[i, 2].imshow(Image.open(row['overlay_saved']))
    axes[i, 2].set_title(f"Overlay — pred {row['predicted_class']} ({row['confidence_percent']:.1f}%)")
    for ax in axes[i]: ax.axis('off')
plt.tight_layout()
plt.show()


## Output folders

- `original/`: selected original test images
- `heatmap/`: resized DeiT last-block CLS-to-patch attention maps
- `overlay/`: attention heatmap blended with the original image
- `Benign/` and `Malignant/`: overlays grouped by **ground-truth** class
- `deit_100_attention_metadata.csv`: true/predicted labels, confidence, correctness and file paths
- `deit_100_attention_summary.csv`: class-wise confidence and correct-prediction summary

**Important:** the selected images are sampled from the test set by ground-truth class, not filtered to only correct predictions. If your paper reports maps only for correctly classified images with confidence above 90%, filter `metadata_df` using `correct == True` and `confidence_percent > 90` and generate/select those separately. Do not imply that attention weights are causal explanations.
